# 📖 Notebook 2: Partitioning and Consumer Groups

Welcome back! In the previous notebook, we learned the basics of producing and consuming messages with Kafka.
Now we'll explore two of Kafka's most powerful features: **partitions** and **consumer groups**.

## 🎯 Learning Objectives

By the end of this notebook, you will understand:

1. **How partitioning enables parallel processing** — splitting work across multiple logs
2. **How partition keys determine message distribution** — controlling which messages go where
3. **What consumer groups are and how they split work** — teams of consumers sharing the load
4. **How to handle hot partitions** — avoiding bottlenecks when one key gets too much traffic

These concepts are essential for building systems that can handle real-world scale.

## 🛠️ Setup

Before running this notebook, make sure you have:

1. **Docker containers running:**
   ```bash
   cd 03-technologies/messaging/kafka
   docker-compose up -d
   ```

2. **Virtual environment activated:**
   ```bash
   cd 03-technologies/messaging/kafka
   uv venv
   source .venv/bin/activate   # On Windows: .venv\Scripts\activate
   uv sync
   ```

3. **Select the `.venv` kernel** in VS Code's kernel picker (top-right corner of the notebook).  
   If the kernel doesn't appear, reload the VS Code window:  
   `Cmd+Shift+P` → **"Reload Window"**

In [ ]:
from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient, NewTopic
import json, time

KAFKA_CONFIG = {'bootstrap.servers': 'localhost:9092'}

admin = AdminClient(KAFKA_CONFIG)
try:
    metadata = admin.list_topics(timeout=5)
    print(f"✅ Connected to Kafka!")
except Exception as e:
    print(f"❌ Cannot connect: {e}")
    print("   Run: cd 03-technologies/messaging/kafka && docker-compose up -d")

## 🧩 Why Partitions?

Imagine the World Cup suddenly scales to **1,000 games running at once**. A single queue would
be completely overwhelmed trying to handle all those events. That's why Kafka uses **partitions**.

### What is a Partition?

A **partition** is an independent, ordered log within a topic. When you create a topic with
multiple partitions, Kafka splits the data across them.

Think of it like **checkout lanes at a grocery store**:
- One lane (one partition) = one cashier processing customers one at a time
- Four lanes (four partitions) = four cashiers working **in parallel**
- More lanes = more throughput!

### How it Looks Inside Kafka

```
                    Topic: "sports-events"
                    ┌──────────────────┐
                    │   4 Partitions    │
                    └──────────────────┘

  Partition 0          Partition 1          Partition 2          Partition 3
  (Broker 1)           (Broker 1)           (Broker 2)           (Broker 3)
  ┌─┬─┬─┬─┬─┐        ┌─┬─┬─┬─┐           ┌─┬─┬─┐              ┌─┬─┬─┬─┬─┬─┐
  │0│1│2│3│4│        │0│1│2│3│           │0│1│2│              │0│1│2│3│4│5│
  └─┴─┴─┴─┴─┘        └─┴─┴─┴─┘           └─┴─┴─┘              └─┴─┴─┴─┴─┴─┘
```

Each partition:
- Has its **own ordered sequence** of messages (offsets 0, 1, 2, ...)
- Can live on a **different broker** (server) for fault tolerance
- Can be read by a **different consumer** for parallelism

> **Key insight:** Messages are ordered **within** a partition, but there's **no ordering
> guarantee across** partitions.

## 🔑 How Partition Keys Work

When you send a message, you can include an optional **key**. Kafka uses this key to decide
which partition the message goes to:

```
partition = hash(key) % num_partitions
```

This is a simple formula with powerful consequences:

| Scenario | What Happens |
|---|---|
| **Same key** → same partition | Guaranteed! If you send all "Brazil vs Germany" events with that key, they always land in the same partition — **in order**. |
| **Different keys** → probably different partitions | The hash spreads keys across partitions evenly (most of the time). |
| **No key** → round-robin | Messages are distributed across partitions in turn. No ordering guarantee at all. |

### When to Use a Key?

- ✅ Use a key when you need **ordering** for related events (e.g., all events for one match)
- ✅ Use a key when consumers need to see **all events for one entity** (e.g., one user's actions)
- ❌ Skip the key when you just want **maximum throughput** and don't care about ordering

Let's see this in action!

In [ ]:
# Create a topic with 4 partitions
topic = NewTopic('sports-events', num_partitions=4, replication_factor=1)
futures = admin.create_topics([topic])
for name, f in futures.items():
    try:
        f.result()
        print(f"✅ Created topic '{name}' with 4 partitions")
    except Exception as e:
        print(f"ℹ️ {name}: {e}")

producer = Producer(KAFKA_CONFIG)
partition_map = {}

def track_delivery(err, msg):
    if err:
        print(f"❌ {err}")
        return
    key = msg.key().decode('utf-8') if msg.key() else 'None'
    p = msg.partition()
    if key not in partition_map:
        partition_map[key] = p
    print(f"  Key '{key}' → Partition {p}")

matches = [
    "Brazil vs Germany", "Argentina vs France", "Spain vs Japan",
    "England vs Senegal", "Portugal vs Morocco", "Netherlands vs USA",
    "Brazil vs Germany", "Argentina vs France", "Spain vs Japan",
    "England vs Senegal", "Portugal vs Morocco", "Netherlands vs USA",
]

print("📨 Sending 12 match events with match name as key...\n")
for match in matches:
    producer.produce(
        'sports-events',
        key=match.encode('utf-8'),
        value=json.dumps({"match": match, "type": "update"}).encode('utf-8'),
        callback=track_delivery,
    )
producer.flush()

print(f"\n📊 Partition assignment:")
for key, partition in sorted(partition_map.items(), key=lambda x: x[1]):
    print(f"  {key:<25} → Partition {partition}")
print(f"\n💡 Same key ALWAYS goes to the same partition!")

## 👥 Consumer Groups Explained

A **consumer group** is a team of consumers that **share the work** of reading from a topic.
Kafka automatically divides the partitions among the consumers in the group.

Think of it like **splitting a pizza** — no two people eat the same slice:

```
    Topic: "sports-events" (4 partitions)

    ┌────────────┐  ┌────────────┐  ┌────────────┐  ┌────────────┐
    │Partition 0  │  │Partition 1  │  │Partition 2  │  │Partition 3  │
    └─────┬──────┘  └─────┬──────┘  └─────┬──────┘  └─────┬──────┘
          │               │               │               │
          │               │               │               │
          ▼               ▼               ▼               ▼
    ┌─────────────────────┐         ┌─────────────────────┐
    │    Consumer 1       │         │    Consumer 2       │
    │  (Partitions 0, 1)  │         │  (Partitions 2, 3)  │
    └─────────────────────┘         └─────────────────────┘
              ╰───────── Consumer Group: "match-processors" ─────────╯
```

### Key Rules of Consumer Groups

| Rule | Explanation |
|---|---|
| **One partition → one consumer** | Within a group, each partition is assigned to exactly ONE consumer. No duplicates! |
| **Consumers > partitions = idle consumers** | If you have 6 consumers but only 4 partitions, 2 consumers will sit idle (waiting as backups). |
| **Consumer dies → rebalancing** | If a consumer crashes, Kafka automatically redistributes its partitions to the remaining consumers. |
| **Different groups are independent** | Each consumer group gets ALL messages. Groups don't interfere with each other. |

### Scaling with Consumer Groups

```
  1 consumer, 4 partitions:     Consumer 1 reads ALL 4 partitions (does all the work)
  2 consumers, 4 partitions:    Each reads 2 partitions (work is split 50/50)
  4 consumers, 4 partitions:    Each reads 1 partition (maximum parallelism!)
  6 consumers, 4 partitions:    4 active + 2 idle (wasting resources)
```

> **Key insight:** The number of partitions sets the **maximum parallelism** for a consumer group.

In [ ]:
import threading

# Send some fresh messages first
producer = Producer(KAFKA_CONFIG)

topic_name = 'group-demo'
t = NewTopic(topic_name, num_partitions=4, replication_factor=1)
fs = admin.create_topics([t])
for n, f in fs.items():
    try:
        f.result()
    except:
        pass

for i in range(20):
    producer.produce(topic_name, key=f"key-{i % 6}".encode(), value=f"message-{i}".encode())
producer.flush()
print(f"✅ Sent 20 messages to '{topic_name}'\n")

time.sleep(1)

results = {1: [], 2: []}

def run_consumer(consumer_id):
    config = {
        **KAFKA_CONFIG,
        'group.id': 'demo-group',
        'auto.offset.reset': 'earliest',
        'session.timeout.ms': 10000,
    }
    c = Consumer(config)
    c.subscribe([topic_name])

    empty = 0
    while empty < 5:
        msg = c.poll(timeout=1.0)
        if msg is None:
            empty += 1
            continue
        if msg.error():
            continue
        empty = 0
        results[consumer_id].append({
            'partition': msg.partition(),
            'value': msg.value().decode(),
        })
    c.close()

t1 = threading.Thread(target=run_consumer, args=(1,))
t2 = threading.Thread(target=run_consumer, args=(2,))
t1.start()
t2.start()
t1.join()
t2.join()

print("📊 Consumer Group Results:")
print("=" * 50)
for cid in [1, 2]:
    partitions = set(m['partition'] for m in results[cid])
    print(f"\n  Consumer {cid}: {len(results[cid])} messages from partitions {sorted(partitions)}")
    for m in results[cid][:3]:
        print(f"    partition={m['partition']}: {m['value']}")
    if len(results[cid]) > 3:
        print(f"    ... and {len(results[cid]) - 3} more")

print(f"\n💡 Each consumer got different partitions — they split the work!")
total = len(results[1]) + len(results[2])
print(f"📊 Total: {total} messages (Consumer 1: {len(results[1])}, Consumer 2: {len(results[2])})")

## 🔀 Multiple Consumer Groups

Different consumer groups are **completely independent**. Each group maintains its own
position (offset) in the topic and gets **ALL** messages.

This is one of Kafka's superpowers — the same stream of data can power **multiple use cases**
without any interference:

```
    Topic: "match-events"
    ┌──────────────────────────────────────┐
    │  event-1  event-2  event-3  event-4  │
    └────┬──────────┬──────────┬───────────┘
         │          │          │
         ▼          ▼          ▼
    ┌─────────┐ ┌──────────┐ ┌───────────┐
    │ Group A  │ │ Group B   │ │ Group C    │
    │ Website  │ │ Push      │ │ Analytics  │
    │ Updaters │ │ Notifs    │ │ Pipeline   │
    │          │ │           │ │            │
    │ Gets ALL │ │ Gets ALL  │ │ Gets ALL   │
    │ messages │ │ messages  │ │ messages   │
    └─────────┘ └──────────┘ └───────────┘
```

This is fundamentally different from traditional message queues (like RabbitMQ), where once a
message is consumed, it's gone. In Kafka, messages persist and can be read by as many groups
as you want.

In [ ]:
topic_name2 = 'multi-group-demo'
t2 = NewTopic(topic_name2, num_partitions=2, replication_factor=1)
fs = admin.create_topics([t2])
for n, f in fs.items():
    try:
        f.result()
    except:
        pass

producer = Producer(KAFKA_CONFIG)
for i in range(5):
    producer.produce(topic_name2, value=f"event-{i}".encode())
producer.flush()
print("✅ Sent 5 messages\n")

time.sleep(1)

def read_all(group_name):
    c = Consumer({**KAFKA_CONFIG, 'group.id': group_name, 'auto.offset.reset': 'earliest'})
    c.subscribe([topic_name2])
    msgs = []
    empty = 0
    while empty < 3:
        msg = c.poll(1.0)
        if msg is None:
            empty += 1
            continue
        if msg.error():
            continue
        empty = 0
        msgs.append(msg.value().decode())
    c.close()
    return msgs

group_a = read_all('website-updaters')
group_b = read_all('notification-senders')

print(f"📊 Group 'website-updaters':      {len(group_a)} messages → {group_a}")
print(f"📊 Group 'notification-senders':   {len(group_b)} messages → {group_b}")
print(f"\n💡 Both groups got ALL messages independently!")

## 🔥 The Hot Partition Problem

Remember that `partition = hash(key) % num_partitions`? This means **all messages with the
same key go to the same partition**. Usually that's great — but what happens when one key
gets way more traffic than others?

### Real-World Example

Imagine you're tracking ad clicks, and you use `ad_id` as the key:

- Most ads get a few clicks per minute
- Then **Nike launches a Super Bowl ad** and gets 10,000 clicks per second
- ALL those clicks go to **one partition** → that partition is overwhelmed!

```
  🔥 Hot Partition!

  Partition 0: ████████████████████████████████ (10,000 msgs/sec)  ← Nike!
  Partition 1: ██ (50 msgs/sec)
  Partition 2: █ (30 msgs/sec)
  Partition 3: ███ (80 msgs/sec)
```

### Strategies to Fix Hot Partitions

| Strategy | How It Works | Trade-off |
|---|---|---|
| **No key (round-robin)** | Skip the key entirely, Kafka distributes evenly | Lose ordering for that entity |
| **Compound key** | Use `ad_id + region` as key (e.g., `nike:US-East`) | Spread load while keeping per-region ordering |
| **Random salting** | Use `ad_id + random_suffix` as key | Best distribution, but harder to aggregate later |

The **compound key** approach is usually the best balance — let's see it in action!

In [ ]:
from collections import Counter

topic_hot = 'hot-partition-demo'
t = NewTopic(topic_hot, num_partitions=4, replication_factor=1)
fs = admin.create_topics([t])
for n, f in fs.items():
    try:
        f.result()
    except:
        pass

producer = Producer(KAFKA_CONFIG)

# Scenario 1: Simple key → hot partition
print("🔥 Scenario 1: Using ad_id as key")
print("-" * 40)
partitions_simple = []

def track_simple(err, msg):
    if not err:
        partitions_simple.append(msg.partition())

ads = ["nike-lebron"] * 15 + ["adidas-messi"] * 3 + ["puma-neymar"] * 2
for ad in ads:
    producer.produce(topic_hot, key=ad.encode(), value=b"click", callback=track_simple)
producer.flush()

counts = Counter(partitions_simple)
for p in sorted(counts):
    bar = "█" * counts[p]
    print(f"  Partition {p}: {bar} ({counts[p]} messages)")
print("  ⚠️ Notice how one partition gets most of the traffic!\n")

# Scenario 2: Compound key → spread the load
print("✅ Scenario 2: Using ad_id + region as compound key")
print("-" * 40)

topic_spread = 'spread-partition-demo'
t2 = NewTopic(topic_spread, num_partitions=4, replication_factor=1)
fs = admin.create_topics([t2])
for n, f in fs.items():
    try:
        f.result()
    except:
        pass

partitions_compound = []

def track_compound(err, msg):
    if not err:
        partitions_compound.append(msg.partition())

import random
regions = ["US-East", "US-West", "Europe", "Asia"]
for _ in range(20):
    ad = "nike-lebron"
    region = random.choice(regions)
    compound_key = f"{ad}:{region}"
    producer.produce(topic_spread, key=compound_key.encode(), value=b"click", callback=track_compound)
producer.flush()

counts2 = Counter(partitions_compound)
for p in sorted(counts2):
    bar = "█" * counts2[p]
    print(f"  Partition {p}: {bar} ({counts2[p]} messages)")
print("  💡 Traffic is much more evenly distributed!")

## 📝 Key Takeaways

Let's recap everything we learned:

- 🛒 **Partitions enable parallel processing** — like checkout lanes at a grocery store, more partitions = more throughput
- 🔑 **Partition key formula:** `hash(key) % num_partitions` — same key always goes to the same partition
- 👥 **Consumer groups split partitions** among members — each partition goes to exactly one consumer in the group
- 🔀 **Different consumer groups are independent** — each group gets ALL messages (great for multiple use cases)
- 🔥 **Watch out for hot partitions** — use compound keys or salting to spread the load
- ⚖️ **More partitions = more parallelism**, but don't go overboard (each partition uses resources on brokers)

### Quick Reference

```
Partitions:      The unit of parallelism in Kafka
Partition Key:   Determines which partition a message goes to
Consumer Group:  A team of consumers sharing the work
Rebalancing:     Automatic redistribution when consumers join/leave
Hot Partition:   One partition getting disproportionate traffic
```

## ⏭️ What's Next?

In the next notebook, we'll explore **exactly-once semantics and delivery guarantees** —
how Kafka ensures your messages are delivered reliably, even when things go wrong.

We'll cover:
- At-most-once, at-least-once, and exactly-once delivery
- Idempotent producers
- Transactional messaging
- How to choose the right guarantee for your use case

See you there! 🚀